# Value Normalization Tutorial

> **Documentation:** For detailed API references and additional examples, see the [Normalization Wiki](../../../wiki/Normalization.md).

This notebook demonstrates the full normalization workflow:

1. **Load** - Load a challenging dataset with messy, inconsistent data
2. **Profile** - Analyze the DataFrame to detect types and patterns
3. **Explore** - Use individual functions to understand specific values
4. **Specify** - Define a normalization specification (schema)
5. **Transform** - Apply transformations to clean the data
6. **Compare** - Review before/after results

The dataset includes edge cases like:
- Scale modifiers: MEO, MEUR, kEUR, millions, billions
- Various unit formats: km, miles, m², sqft, °C, °F
- Country formats: names, alpha-2, alpha-3 codes
- Phone numbers: international formats with various separators
- Emails: uppercase, whitespace issues
- Standard numbers: VAT, IBAN (valid and invalid)

In [1]:
# Imports
from pathlib import Path

from PyDI.io import load_csv
from PyDI.normalization import (
    # Profiling
    profile_dataframe,
    # Specification and transformation
    NormalizationSpec,
    transform_dataframe,
    normalize_dataframe,
    # Individual functions
    parse_quantity,
    convert_units,
    detect_scale_modifier,
    parse_scaled_number,
    normalize_country,
    normalize_currency,
    parse_phone,
    format_phone,
    validate_email,
    normalize_email,
    is_valid_iban,
    is_valid_vat,
)

## Step 1: Load the Challenging Dataset

In [2]:
# Load dataset using PyDI's load_csv
df = load_csv(Path("challenging_dataset.csv"), name="companies")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Shape: 20 rows x 13 columns


,company_name,revenue,employees,country,headquarters_phone,contact_email,distance_to_port,warehouse_area,vat_number,iban,temperature_range,annual_growth,currency_code
0,Acme Industries GmbH,5.2 MEO,150,Germany,+49 30 12345678,INFO@ACME-INDUSTRIES.DE,45 km,2500 m²,DE123456789,DE89370400440532013000,15°C to 25°C,12.5%,Euro
1,TechStart Ltd,2.5 MEUR,85,United Kingdom,+44 20 7946 0958,contact@techstart.co.uk,100 miles,50000 sq ft,GB123456789,GB82WEST12345698765432,-5 to 35 celsius,8.3 percent,GBP
2,Global Finance SA,750 kEUR,42,Portugal,00351 21 1234567,sales@globalfinance.pt,3.5 kilometers,1200 m2,PT501234567,PT50000201231234567890154,20-30 °C,+5.7%,EUR
3,DataSolutions Inc,10M,$500k marketing,DEU,1-555-123-4567,SUPPORT@datasolutions.com,2500 m,10000 square feet,invalid-vat,invalid-iban,68°F - 86°F,15 percent growth,USD
4,Nordic Ventures AB,1.2 billion SEK,320,Sweden,+46 8 123 45 67,info@nordicventures.se,75 km,3000 m²,SE556789012301,SE4550000000058398257466,10 celsius,twenty percent,Swedish Krona


## Step 2: Profile the DataFrame

Profiling detects:
- Data types and patterns
- Units and scale modifiers
- Standard number formats
- Country/currency codes

In [3]:
profile = profile_dataframe(df)
print(profile.summary())

DataFrame Profile: 20 rows, 13 columns

company_name:
  Type: text
  Samples: ['Acme Industries GmbH', 'TechStart Ltd', 'Global Finance SA']

revenue:
  Type: scaled_number
  Samples: ['5.2 MEO', '2.5 MEUR', '750 kEUR']
  Scale modifiers: {'modifiers_detected': {'billion': 4, 'M': 3, 'million': 3, 'MEO': 2, 'MEUR': 2}, 'coverage': 0.8}
  Suggestion: Consider expanding scale modifiers (e.g., '5 MEO' → 5000000)

employees:
  Type: text
  Samples: ['150', '  85  ', '42']

country:
  Type: country
  Samples: ['Germany', 'United Kingdom', 'Portugal']
  Country formats: {'formats_detected': {'name': 11, 'alpha_3': 6, 'alpha_2': 1}, 'coverage': 0.9}
  Suggestion: Consider normalizing to ISO 3166 alpha-2 codes

headquarters_phone:
  Type: phone
  Samples: ['+49 30 12345678', '+44 20 7946 0958', '00351 21 1234567']
  Suggestion: Consider normalizing to E.164 format

contact_email:
  Type: email
  Samples: ['INFO@ACME-INDUSTRIES.DE', 'contact@techstart.co.uk', 'sales@globalfinance.pt']
  Suggest

## Step 3: Explore Specific Values

Use individual normalization functions to understand the data.

In [4]:
# Revenue: parse scale modifiers (MEO, MEUR, millions, etc.)
print("Revenue values parsed:")
for val in df["revenue"].head(8):
    result = parse_scaled_number(str(val))
    mod = detect_scale_modifier(str(val))
    if result:
        print(f"  '{val}' -> {result[0]:,.0f} (modifier: {mod.name if mod else 'none'})")
    else:
        print(f"  '{val}' -> could not parse")

Revenue values parsed:
  '5.2 MEO' -> 5,200,000 (modifier: MEO)
  '2.5 MEUR' -> 2,500,000 (modifier: MEUR)
  '750 kEUR' -> 750,000 (modifier: kEUR)
  '10M' -> 10,000,000 (modifier: M)
  '1.2 billion SEK' -> 1,200,000,000 (modifier: billion)
  '500 TEUR' -> 500,000 (modifier: TEUR)
  '¥50 million' -> 50,000,000 (modifier: million)
  '3.8 MEO' -> 3,800,000 (modifier: MEO)


In [5]:
# Distance: parse units and convert to km
print("Distance values converted to km:")
for val in df["distance_to_port"].head(8):
    parsed = parse_quantity(str(val))
    if parsed:
        km = convert_units(parsed.magnitude, parsed.unit, "km")
        print(f"  '{val}' -> {parsed.magnitude} {parsed.unit} = {km:.2f} km")
    else:
        print(f"  '{val}' -> could not parse")

Distance values converted to km:
  '45 km' -> 45.0 km = 45.00 km
  '100 miles' -> 100.0 mi = 160.93 km
  '3.5 kilometers' -> 3.5 km = 3.50 km
  '2500 m' -> 2500.0 m = 2.50 km
  '75 km' -> 75.0 km = 75.00 km
  '12.5 km' -> 12.5 km = 12.50 km
  '500 kilometers' -> 500.0 km = 500.00 km
  '85 km' -> 85.0 km = 85.00 km


In [6]:
# Country: normalize to standard codes
print("Country normalization:")
for val in df["country"].head(10):
    alpha2 = normalize_country(str(val), output_format="alpha_2")
    name = normalize_country(str(val), output_format="name")
    print(f"  '{val}' -> alpha_2: {alpha2}, name: {name}")

Country normalization:
  'Germany' -> alpha_2: DE, name: Germany
  'United Kingdom' -> alpha_2: GB, name: United Kingdom
  'Portugal' -> alpha_2: PT, name: Portugal
  'DEU' -> alpha_2: DE, name: Germany
  'Sweden' -> alpha_2: SE, name: Sweden
  'Nederland' -> alpha_2: None, name: None
  'China' -> alpha_2: CN, name: China
  'ESP' -> alpha_2: ES, name: Spain
  'CH' -> alpha_2: CH, name: Switzerland
  'United States of America' -> alpha_2: US, name: United States


In [7]:
# Phone numbers: parse and format
print("Phone number parsing:")
for val in df["headquarters_phone"].head(6):
    info = parse_phone(str(val))
    if info:
        e164 = format_phone(str(val), format="E164")
        print(f"  '{val}' -> country: {info.country_code}, E164: {e164}")
    else:
        print(f"  '{val}' -> invalid")

Phone number parsing:
  '+49 30 12345678' -> country: 49, E164: +493012345678
  '+44 20 7946 0958' -> country: 44, E164: +442079460958
  '00351 21 1234567' -> country: 351, E164: +351211234567
  '1-555-123-4567' -> country: None, E164: None
  '+46 8 123 45 67' -> country: 46, E164: +4681234567
  '+31 20 123 4567' -> country: 31, E164: +31201234567


## Step 4: Define Normalization Specification

Create a spec that defines how each column should be normalized.

In [8]:
spec = NormalizationSpec()

# Company name: strip whitespace, title case
spec.set_column("company_name", strip_whitespace=True, case="title")

# Revenue: expand scale modifiers (MEO -> 1000000, etc.)
spec.set_column("revenue", expand_scale_modifiers=True, output_type="float")

# Employees: convert to integer (strip whitespace first)
spec.set_column("employees", strip_whitespace=True, output_type="int")

# Country: normalize to ISO alpha-2 codes
spec.set_column("country", country_format="alpha_2")

# Phone: normalize to E.164 format
spec.set_column("headquarters_phone", phone_format="e164", phone_default_region="US")

# Email: normalize (lowercase, strip whitespace)
spec.set_column("contact_email", normalize_email=True)

# Distance: convert all to kilometers
spec.set_column("distance_to_port", target_unit="km", output_type="float")

# Area: convert to square meters
spec.set_column("warehouse_area", target_unit="m**2", output_type="float")

# VAT: format standardized
spec.set_column("vat_number", stdnum_format=True, on_failure="null")

# IBAN: format standardized
spec.set_column("iban", stdnum_format=True, on_failure="null")

# Temperature: convert to celsius (note: ranges are tricky, this will try to parse)
#spec.set_column("temperature_range", target_unit="degC", output_type="float")

# Annual growth: extract percentage as float
spec.set_column("annual_growth", convert_percentage="to_decimal", output_type="float")

# Currency: normalize to ISO alpha-3
spec.set_column("currency_code", currency_format="alpha_3")

NormalizationSpec(columns={'company_name': ColumnSpec(output_type='keep', on_failure='keep', target_unit=None, expand_scale_modifiers=False, convert_percentage=None, country_format=None, currency_format=None, phone_format=None, phone_default_region='US', normalize_email=False, stdnum_format=False, case='title', strip_whitespace=True), 'revenue': ColumnSpec(output_type='float', on_failure='keep', target_unit=None, expand_scale_modifiers=True, convert_percentage=None, country_format=None, currency_format=None, phone_format=None, phone_default_region='US', normalize_email=False, stdnum_format=False, case=None, strip_whitespace=False), 'employees': ColumnSpec(output_type='int', on_failure='keep', target_unit=None, expand_scale_modifiers=False, convert_percentage=None, country_format=None, currency_format=None, phone_format=None, phone_default_region='US', normalize_email=False, stdnum_format=False, case=None, strip_whitespace=True), 'country': ColumnSpec(output_type='keep', on_failure='kee

## Step 5: Transform the Data

Apply the specification to transform the DataFrame.

In [9]:
# Execute the transformation
result = transform_dataframe(df, spec)

print(f"Transformation Summary:")
print(f"  Total values transformed: {result.total_transformed}")
print(f"  Total failures: {result.total_failed}")

# Get the normalized DataFrame
normalized_df = result.dataframe

Transformation Summary:
  Total values transformed: 172
  Total failures: 36


In [10]:
# Show per-column transformation results and errors
for col_name, col_result in result.columns.items():
    status = "✓" if col_result.values_failed == 0 else "✗"
    print(f"{status} {col_name}: {col_result.values_transformed} transformed, {col_result.values_failed} failed")
    if col_result.errors:
        for error in col_result.errors:
            print(f"    Error: {error}")

✓ company_name: 14 transformed, 0 failed
✓ revenue: 20 transformed, 0 failed
✓ employees: 1 transformed, 0 failed
✗ country: 18 transformed, 2 failed
    Error: Row 5: Could not transform 'Nederland'
    Error: Row 12: Could not transform 'Brasil'
✓ headquarters_phone: 20 transformed, 0 failed
✓ contact_email: 20 transformed, 0 failed
✓ distance_to_port: 20 transformed, 0 failed
✗ warehouse_area: 17 transformed, 3 failed
    Error: Row 2: Could not transform '1200 m2'
    Error: Row 6: Could not transform '25000 sqm'
    Error: Row 9: Could not transform '100000 sqft'
✗ vat_number: 0 transformed, 15 failed
    Error: Row 0: Could not transform 'DE123456789'
    Error: Row 1: Could not transform 'GB123456789'
    Error: Row 2: Could not transform 'PT501234567'
    Error: Row 3: Could not transform 'invalid-vat'
    Error: Row 4: Could not transform 'SE556789012301'
    Error: Row 5: Could not transform 'NL123456789B01'
    Error: Row 7: Could not transform 'ES12345678A'
    Error: Row 8

## Step 6: Compare Before vs After

In [11]:
# Compare original vs normalized side by side (original and new column next to each other)
import pandas as pd

comparison_cols = []
for col in df.columns:
    comparison_cols.append((f"{col}_orig", df[col]))
    comparison_cols.append((f"{col}_norm", normalized_df[col]))

comparison = pd.DataFrame(dict(comparison_cols))
comparison

,company_name_orig,company_name_norm,revenue_orig,revenue_norm,employees_orig,employees_norm,country_orig,country_norm,headquarters_phone_orig,headquarters_phone_norm,...,vat_number_orig,vat_number_norm,iban_orig,iban_norm,temperature_range_orig,temperature_range_norm,annual_growth_orig,annual_growth_norm,currency_code_orig,currency_code_norm
0,Acme Industries GmbH,Acme Industries Gmbh,5.2 MEO,5.200000e+06,150,150,Germany,DE,+49 30 12345678,+493012345678,...,DE123456789,NaN,DE89370400440532013000,DE89 3704 0044 0532 0130 00,15°C to 25°C,15°C to 25°C,12.5%,0.125,Euro,EUR
1,TechStart Ltd,Techstart Ltd,2.5 MEUR,2.500000e+06,85,85,United Kingdom,GB,+44 20 7946 0958,+442079460958,...,GB123456789,NaN,GB82WEST12345698765432,GB82 WEST 1234 5698 7654 32,-5 to 35 celsius,-5 to 35 celsius,8.3 percent,8.3 percent,GBP,GBP
2,Global Finance SA,Global Finance Sa,750 kEUR,7.500000e+05,42,42,Portugal,PT,00351 21 1234567,+351211234567,...,PT501234567,NaN,PT50000201231234567890154,PT50 0002 0123 1234 5678 9015 4,20-30 °C,20-30 °C,+5.7%,0.057,EUR,EUR
3,DataSolutions Inc,Datasolutions Inc,10M,1.000000e+07,$500k marketing,$500k marketing,DEU,DE,1-555-123-4567,+15551234567,...,invalid-vat,NaN,invalid-iban,None,68°F - 86°F,68°F - 86°F,15 percent growth,15 percent growth,USD,USD
4,Nordic Ventures AB,Nordic Ventures Ab,1.2 billion SEK,1.200000e+09,320,320,Sweden,SE,+46 8 123 45 67,+4681234567,...,SE556789012301,NaN,SE4550000000058398257466,SE45 5000 0000 0583 9825 7466,10 celsius,10 celsius,twenty percent,twenty percent,Swedish Krona,SEK
5,EuroTrade BV,Eurotrade Bv,500 TEUR,5.000000e+05,28,28,Nederland,Nederland,+31 20 123 4567,+31201234567,...,NL123456789B01,NaN,NL91ABNA0417164300,NL91 ABNA 0417 1643 00,18°C,18°C,7.2%,0.072,EUR,EUR
6,Asian Manufacturing Co,Asian Manufacturing Co,¥50 million,5.000000e+07,1200,1200,China,CN,+86 21 1234 5678,+862112345678,...,NaN,NaN,NaN,NaN,NaN,NaN,CNY,CNY,NaN,NaN
7,Iberian Partners SL,Iberian Partners Sl,3.8 MEO,3.800000e+06,95,95,ESP,ES,+34 91 123 4567,+34911234567,...,ES12345678A,NaN,ES9121000418450200051332,ES91 2100 0418 4502 0005 1332,22°C - 28°C,22°C - 28°C,9.1%,0.091,Euro,EUR
8,Swiss Precision AG,Swiss Precision Ag,8.5 MCHF,8.500000e+00,180,180,CH,CH,+41 44 123 45 67,+41441234567,...,CHE-123.456.789,NaN,CH9300762011623852957,CH93 0076 2011 6238 5295 7,5 to 20 degrees C,5 to 20 degrees C,11.4%,0.114,CHF,CHF
9,American Logistics LLC,American Logistics Llc,25M,2.500000e+07,450,450,United States of America,US,(800) 555-0199,+18005550199,...,invalid,NaN,US12345678901234567890,None,50-90°F,50-90°F,6.8%,0.068,Dollar,Dollar


## Auto-Normalize the Full Dataset

Apply auto-normalization to the entire challenging dataset in one line.

In [12]:
# Auto-normalize the full challenging dataset
auto_full = normalize_dataframe(df, auto=True)
print(f"Auto-normalized {len(auto_full)} rows")
auto_full.head(10)

Auto-normalized 20 rows


,company_name,revenue,employees,country,headquarters_phone,contact_email,distance_to_port,warehouse_area,vat_number,iban,temperature_range,annual_growth,currency_code
0,Acme Industries GmbH,5.200000e+06,150,DE,+493012345678,info@acme-industries.de,45.0000,2500.0,DE123456789,DE89370400440532013000,15°C to 25°C,12.5,EUR
1,TechStart Ltd,2.500000e+06,85,GB,+442079460958,contact@techstart.co.uk,160.9344,4645.152,GB123456789,GB82WEST12345698765432,-5 to 35 celsius,8.3,GBP
2,Global Finance SA,7.500000e+05,42,PT,+351211234567,sales@globalfinance.pt,3.5000,1200 m2,PT501234567,PT50000201231234567890154,20-30 °C,5.7,EUR
3,DataSolutions Inc,1.000000e+07,$500k marketing,DE,+15551234567,support@datasolutions.com,2.5000,929.0304,invalid-vat,invalid-iban,68°F - 86°F,15 percent growth,USD
4,Nordic Ventures AB,1.200000e+09,320,SE,+4681234567,info@nordicventures.se,75.0000,3000.0,SE556789012301,SE4550000000058398257466,10 celsius,twenty percent,SEK
5,EuroTrade BV,5.000000e+05,28,Nederland,+31201234567,handel@eurotrade.nl,12.5000,800.0,NL123456789B01,NL91ABNA0417164300,18°C,7.2,EUR
6,Asian Manufacturing Co,5.000000e+07,1200,CN,+862112345678,contact@asianmfg.cn,500.0000,25000 sqm,NaN,NaN,NaN,CNY,NaN
7,Iberian Partners SL,3.800000e+06,95,ES,+34911234567,partners@iberian.es,85.0000,4500.0,ES12345678A,ES9121000418450200051332,22°C - 28°C,9.1,EUR
8,Swiss Precision AG,8.500000e+00,180,CH,+41441234567,info@swissprecision.ch,15.0000,6000.0,CHE-123.456.789,CH9300762011623852957,5 to 20 degrees C,11.4,CHF
9,American Logistics LLC,2.500000e+07,450,US,+18005550199,logistics@americanlog.com,241.4016,100000 sqft,invalid,US12345678901234567890,50-90°F,6.8,Dollar


In [13]:
# Compare original vs auto-normalized side by side (original and new column next to each other)
comparison_cols = []
for col in df.columns:
    comparison_cols.append((f"{col}_orig", df[col]))
    comparison_cols.append((f"{col}_auto", auto_full[col]))

comparison_full = pd.DataFrame(dict(comparison_cols))
comparison_full

,company_name_orig,company_name_auto,revenue_orig,revenue_auto,employees_orig,employees_auto,country_orig,country_auto,headquarters_phone_orig,headquarters_phone_auto,...,vat_number_orig,vat_number_auto,iban_orig,iban_auto,temperature_range_orig,temperature_range_auto,annual_growth_orig,annual_growth_auto,currency_code_orig,currency_code_auto
0,Acme Industries GmbH,Acme Industries GmbH,5.2 MEO,5.200000e+06,150,150,Germany,DE,+49 30 12345678,+493012345678,...,DE123456789,DE123456789,DE89370400440532013000,DE89370400440532013000,15°C to 25°C,15°C to 25°C,12.5%,12.5,Euro,EUR
1,TechStart Ltd,TechStart Ltd,2.5 MEUR,2.500000e+06,85,85,United Kingdom,GB,+44 20 7946 0958,+442079460958,...,GB123456789,GB123456789,GB82WEST12345698765432,GB82WEST12345698765432,-5 to 35 celsius,-5 to 35 celsius,8.3 percent,8.3,GBP,GBP
2,Global Finance SA,Global Finance SA,750 kEUR,7.500000e+05,42,42,Portugal,PT,00351 21 1234567,+351211234567,...,PT501234567,PT501234567,PT50000201231234567890154,PT50000201231234567890154,20-30 °C,20-30 °C,+5.7%,5.7,EUR,EUR
3,DataSolutions Inc,DataSolutions Inc,10M,1.000000e+07,$500k marketing,$500k marketing,DEU,DE,1-555-123-4567,+15551234567,...,invalid-vat,invalid-vat,invalid-iban,invalid-iban,68°F - 86°F,68°F - 86°F,15 percent growth,15 percent growth,USD,USD
4,Nordic Ventures AB,Nordic Ventures AB,1.2 billion SEK,1.200000e+09,320,320,Sweden,SE,+46 8 123 45 67,+4681234567,...,SE556789012301,SE556789012301,SE4550000000058398257466,SE4550000000058398257466,10 celsius,10 celsius,twenty percent,twenty percent,Swedish Krona,SEK
5,EuroTrade BV,EuroTrade BV,500 TEUR,5.000000e+05,28,28,Nederland,Nederland,+31 20 123 4567,+31201234567,...,NL123456789B01,NL123456789B01,NL91ABNA0417164300,NL91ABNA0417164300,18°C,18°C,7.2%,7.2,EUR,EUR
6,Asian Manufacturing Co,Asian Manufacturing Co,¥50 million,5.000000e+07,1200,1200,China,CN,+86 21 1234 5678,+862112345678,...,NaN,NaN,NaN,NaN,NaN,NaN,CNY,CNY,NaN,NaN
7,Iberian Partners SL,Iberian Partners SL,3.8 MEO,3.800000e+06,95,95,ESP,ES,+34 91 123 4567,+34911234567,...,ES12345678A,ES12345678A,ES9121000418450200051332,ES9121000418450200051332,22°C - 28°C,22°C - 28°C,9.1%,9.1,Euro,EUR
8,Swiss Precision AG,Swiss Precision AG,8.5 MCHF,8.500000e+00,180,180,CH,CH,+41 44 123 45 67,+41441234567,...,CHE-123.456.789,CHE-123.456.789,CH9300762011623852957,CH9300762011623852957,5 to 20 degrees C,5 to 20 degrees C,11.4%,11.4,CHF,CHF
9,American Logistics LLC,American Logistics LLC,25M,2.500000e+07,450,450,United States of America,US,(800) 555-0199,+18005550199,...,invalid,invalid,US12345678901234567890,US12345678901234567890,50-90°F,50-90°F,6.8%,6.8,Dollar,Dollar


## Summary

The normalization workflow:

```python
# 1. Profile your data
profile = profile_dataframe(df)

# 2. Define normalization spec
spec = NormalizationSpec()
spec.set_column("revenue", expand_scale_modifiers=True)
spec.set_column("country", country_format="alpha_2")

# 3. Transform
result = transform_dataframe(df, spec)
normalized_df = result.dataframe

# Or use auto-normalization
normalized_df = normalize_dataframe(df, auto=True)
```